***NEXPAY***

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine
from dotenv import load_dotenv

import os

In [5]:
load_dotenv()

user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST", "localhost")
port = os.getenv("POSTGRES_PORT", "5432")
database = os.getenv("POSTGRES_DB", "nexapay_analytics")

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
)

print("PostgreSQL connection successful!")

PostgreSQL connection successful!


In [6]:
query = """
SELECT COUNT(*) AS total_customers
FROM customers;
"""

result = pd.read_sql(query, engine)

result

,total_customers
0,120000


Extracting Customers Data

In [7]:
customers = pd.read_sql(
    """
    SELECT *
    FROM customers;
    """,
    engine
)

customers.head()

,customer_id,customer_tenure_months,age,city,state,customer_segment,income_band,kyc_status,acquisition_channel,signup_date,risk_band
0,1,11,40,Chennai,Tamil Nadu,Mass,50K-100K,Verified,Digital,2025-01-31,Medium
1,2,0,54,Pune,Maharashtra,Affluent,250K+,Verified,Digital,2025-12-30,Low
2,3,20,72,Hyderabad,Telangana,Affluent,100K-250K,Verified,Digital,2024-05-10,Medium
3,4,5,68,Mumbai,Maharashtra,Mass,25K-50K,Verified,Branch,2025-07-18,Medium
4,5,11,65,Ahmedabad,Gujarat,Mass,25K-50K,Verified,Digital,2025-02-04,Low


In [8]:
pd.read_sql(
    """
    SELECT COUNT(*) AS row_count
    FROM transactions;
    """,
    engine
)

,row_count
0,1500000


Checking Row Counts

In [9]:
pd.read_sql(
    """
    SELECT COUNT(*) AS row_count
    FROM transactions;
    """,
    engine
)

,row_count
0,1500000


Transactions Sample Data

In [10]:
transactions_sample = pd.read_sql(
    """
    SELECT *
    FROM transactions
    LIMIT 1000;
    """,
    engine
)

transactions_sample.head()

,transaction_id,transaction_timestamp,customer_id,account_id,merchant_id,transaction_type,channel,payment_method,amount_inr,transaction_status,fraud_score,fraud_flag,fee_inr,currency,device_type,customer_initiated
0,1,2025-12-11 14:56:05.428836,61073,55326,28852,Bill Payment,POS,Wallet,2774.62,Success,50.0,0,11.51,INR,iOS,1
1,2,2025-11-07 17:21:16.775986,76164,4351,11835,Purchase,POS,UPI,1924.75,Success,35.0,0,27.90,INR,Web,1
2,3,2025-12-15 20:02:20.653058,115702,165518,27244,Purchase,POS,Net Banking,1195.99,Success,67.0,0,19.01,INR,POS Terminal,1
3,4,2025-03-20 02:47:52.416736,14136,153259,21399,Transfer,ATM,Bank Transfer,587.16,Success,17.0,0,5.21,INR,Android,1
4,5,2025-08-08 01:23:18.880804,68589,168848,2266,Purchase,Web,UPI,845.30,Success,44.0,0,14.74,INR,ATM,1


In [11]:
transactions_sample.dtypes

transaction_id                    int64
transaction_timestamp    datetime64[us]
customer_id                       int64
account_id                        int64
merchant_id                       int64
transaction_type                    str
channel                             str
payment_method                      str
amount_inr                      float64
transaction_status                  str
fraud_score                     float64
fraud_flag                        int64
fee_inr                         float64
currency                            str
device_type                         str
customer_initiated                int64
dtype: object

In [12]:
transactions_sample.isnull().sum()

transaction_id           0
transaction_timestamp    0
customer_id              0
account_id               0
merchant_id              0
transaction_type         0
channel                  0
payment_method           0
amount_inr               0
transaction_status       0
fraud_score              0
fraud_flag               0
fee_inr                  0
currency                 0
device_type              0
customer_initiated       0
dtype: int64

**Validating Foreign keys**

Transaction--Merchants 

In [13]:
pd.read_sql(
    """
    SELECT COUNT(*) AS invalid_merchant_transactions
    FROM transactions as t
    LEFT JOIN merchants  as m
        ON t.merchant_id = m.merchant_id
    WHERE m.merchant_id IS NULL;
    """,
    engine
)

,invalid_merchant_transactions
0,0


Transaction--Customers

In [14]:
pd.read_sql(
    """
    SELECT COUNT(*) as invalid_customer_transactions
    FROM transactions  as t
    LEFT JOIN customers as c
        ON t.customer_id = c.customer_id
    WHERE c.customer_id IS NULL;
    """,
    engine
)

,invalid_customer_transactions
0,0


*Transaction--Accounts*

In [15]:
pd.read_sql(
    """
    SELECT COUNT(*) as orphan_transactions
    FROM transactions as t

    LEFT JOIN accounts as a
        ON t.account_id = a.account_id

    WHERE a.account_id IS NULL;
    """,
    engine
)

,orphan_transactions
0,0


*ChargeBacks--Transactions*

In [16]:
pd.read_sql(
    """
    SELECT COUNT(*) as orphan_chargebacks
    FROM chargebacks as cb

    LEFT JOIN transactions as t
        ON cb.transaction_id = t.transaction_id

    WHERE t.transaction_id IS NULL;
    """,
    engine
)

,orphan_chargebacks
0,0


**Validate Categorical Values**

*Transaction Status*

In [17]:
pd.read_sql(
    """
    SELECT
        transaction_status,
        COUNT(*) AS transactions
    FROM transactions
    GROUP BY transaction_status
    ORDER BY transactions DESC;
    """,
    engine
)

,transaction_status,transactions
0,Success,1220907
1,Failed,144921
2,Pending,89279
3,Reversed,44893


*Payment Methods*

In [18]:
pd.read_sql(
    """
    SELECT
        payment_method,
        COUNT(*) AS transactions
    FROM transactions
    GROUP BY payment_method
    ORDER BY transactions DESC;
    """,
    engine
)

,payment_method,transactions
0,UPI,540096
1,Card,405582
2,Net Banking,255009
3,Bank Transfer,179408
4,Wallet,119905


In [19]:
transactions_sample["payment_method"] = (
    transactions_sample["payment_method"]
    .str.strip()
    .str.title()
)

*Missing Values in Transactions*

In [20]:
missing_transactions = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) - COUNT(transaction_id)
            AS missing_transaction_id,

        COUNT(*) - COUNT(customer_id)
            AS missing_customer_id,

        COUNT(*) - COUNT(account_id)
            AS missing_account_id,

        COUNT(*) - COUNT(merchant_id)
            AS missing_merchant_id,

        COUNT(*) - COUNT(amount_inr)
            AS missing_amount,

        COUNT(*) - COUNT(transaction_timestamp)
            AS missing_timestamp,

        COUNT(*) - COUNT(transaction_status)
            AS missing_status,

        COUNT(*) - COUNT(payment_method)
            AS missing_payment_method,

        COUNT(*) - COUNT(channel)
            AS missing_channel

    FROM transactions;
    """,
    engine
)

missing_transactions

,total_rows,missing_transaction_id,missing_customer_id,missing_account_id,missing_merchant_id,missing_amount,missing_timestamp,missing_status,missing_payment_method,missing_channel
0,1500000,0,0,0,0,0,0,0,0,0


**Duplicates Validation**

*TransactionID*

In [21]:
duplicate_transactions = pd.read_sql(
    """
    SELECT
        transaction_id,
        COUNT(*) AS occurrences

    FROM transactions

    GROUP BY transaction_id

    HAVING COUNT(*) > 1

    ORDER BY occurrences DESC;
    """,
    engine
)

duplicate_transactions

,transaction_id,occurrences


*CustomerID*

In [22]:
pd.read_sql(
    """
    SELECT
        customer_id,
        COUNT(*) AS occurrences
    FROM customers
    GROUP BY customer_id
    HAVING COUNT(*) > 1;
    """,
    engine
)

,customer_id,occurrences


*AccountID*

In [23]:
pd.read_sql(
    """
    SELECT
        account_id,
        COUNT(*) AS occurrences
    FROM accounts
    GROUP BY account_id
    HAVING COUNT(*) > 1;
    """,
    engine
)

,account_id,occurrences


**Customer Data Validation**

*Customer Segements*

In [24]:
pd.read_sql(
    """
    SELECT
        customer_segment,
        COUNT(*) AS customers
    FROM customers
    GROUP BY customer_segment
    ORDER BY customers DESC;
    """,
    engine
)

,customer_segment,customers
0,Mass,62338
1,Affluent,33690
2,Premium,18026
3,SME,5946


*Customer Risk Bands*

In [25]:
pd.read_sql(
    """
    SELECT
        risk_band,
        COUNT(*) AS customers
    FROM customers
    GROUP BY risk_band
    ORDER BY customers DESC;
    """,
    engine
)

,risk_band,customers
0,Low,74438
1,Medium,37057
2,High,8505


*Customer AGE*

In [26]:
pd.read_sql(
    """
    SELECT
        MIN(age) AS minimum_age,
        MAX(age) AS maximum_age
    FROM customers;
    """,
    engine
)

,minimum_age,maximum_age
0,18,80


In [27]:
pd.read_sql(
    """
    SELECT COUNT(*) AS invalid_age
    FROM customers
    WHERE age < 18
       OR age > 100;
    """,
    engine
)

,invalid_age
0,0


***ANALYTICAL COLUMNS***

*Customers Column*

In [28]:
customers = pd.read_sql(
    """
    SELECT *
    FROM customers;
    """,
    engine
)

In [29]:
customer_metrics = pd.read_sql(
    """
    SELECT
        customer_id,
        COUNT(*) AS total_transactions,
        SUM(amount_inr) AS total_transaction_value,
        AVG(amount_inr) AS average_transaction_value,
        SUM(CASE WHEN transaction_status = 'Success' THEN 1 ELSE 0 END) AS successful_transactions,
        SUM(CASE WHEN transaction_status = 'Failed' THEN 1 ELSE 0 END) AS failed_transactions,
        SUM(fraud_flag) AS fraud_transactions
    FROM transactions
    GROUP BY customer_id;
    """,
    engine
)

In [30]:
customer_analysis = customers.merge(
    customer_metrics,
    on="customer_id",
    how="left"
)

In [31]:
customer_analysis["transaction_success_rate"] = (
    customer_analysis["successful_transactions"]
    / customer_analysis["total_transactions"]
    * 100
)

customer_analysis["transaction_failure_rate"] = (
    customer_analysis["failed_transactions"]
    / customer_analysis["total_transactions"]
    * 100
)

customer_analysis["fraud_rate"] = (
    customer_analysis["fraud_transactions"]
    / customer_analysis["total_transactions"]
    * 100
)

In [32]:
customer_analysis.head()

,customer_id,customer_tenure_months,age,city,state,customer_segment,income_band,kyc_status,acquisition_channel,signup_date,risk_band,total_transactions,total_transaction_value,average_transaction_value,successful_transactions,failed_transactions,fraud_transactions,transaction_success_rate,transaction_failure_rate,fraud_rate
0,1,11,40,Chennai,Tamil Nadu,Mass,50K-100K,Verified,Digital,2025-01-31,Medium,32.0,137793.89,4306.059063,21.0,6.0,0.0,65.625000,18.750000,0.0
1,2,0,54,Pune,Maharashtra,Affluent,250K+,Verified,Digital,2025-12-30,Low,7.0,17695.15,2527.878571,5.0,2.0,0.0,71.428571,28.571429,0.0
2,3,20,72,Hyderabad,Telangana,Affluent,100K-250K,Verified,Digital,2024-05-10,Medium,29.0,80422.99,2773.206552,21.0,2.0,0.0,72.413793,6.896552,0.0
3,4,5,68,Mumbai,Maharashtra,Mass,25K-50K,Verified,Branch,2025-07-18,Medium,25.0,278110.33,11124.413200,21.0,2.0,0.0,84.000000,8.000000,0.0
4,5,11,65,Ahmedabad,Gujarat,Mass,25K-50K,Verified,Digital,2025-02-04,Low,12.0,57457.95,4788.162500,9.0,2.0,0.0,75.000000,16.666667,0.0


*Account-Analysis*

In [33]:
accounts = pd.read_sql(
    """
    SELECT *
    FROM accounts;
    """,
    engine
)

In [34]:
account_metrics = pd.read_sql(
    """
    SELECT
        account_id,
        COUNT(*) AS total_transactions,
        SUM(amount_inr) AS total_transaction_value,
        AVG(amount_inr) AS average_transaction_value,
        SUM(CASE WHEN transaction_status = 'Success' THEN 1 ELSE 0 END) AS successful_transactions,
        SUM(CASE WHEN transaction_status = 'Failed' THEN 1 ELSE 0 END) AS failed_transactions
    FROM transactions
    GROUP BY account_id;
    """,
    engine
)

In [35]:
account_analysis = accounts.merge(
    account_metrics,
    on="account_id",
    how="left"
)

In [36]:
account_analysis["success_rate"] = (
    account_analysis["successful_transactions"]
    / account_analysis["total_transactions"]
    * 100
)

account_analysis["failure_rate"] = (
    account_analysis["failed_transactions"]
    / account_analysis["total_transactions"]
    * 100
)

In [37]:
account_analysis.head()

,account_id,customer_id,account_type,account_status,opened_date,current_balance_inr,credit_limit_inr,total_transactions,total_transaction_value,average_transaction_value,successful_transactions,failed_transactions,success_rate,failure_rate
0,1,6620,Credit,Active,2024-02-17,45854.87,98373.18,10.0,61388.24,6138.824000,9.0,0.0,90.000000,0.000000
1,2,10369,Savings,Active,2024-11-23,128742.19,0.00,5.0,5733.69,1146.738000,5.0,0.0,100.000000,0.000000
2,3,28137,Salary,Active,2023-03-17,51998.54,0.00,12.0,21859.60,1821.633333,10.0,0.0,83.333333,0.000000
3,4,9971,Savings,Active,2025-07-30,22892.15,0.00,9.0,20258.12,2250.902222,8.0,1.0,88.888889,11.111111
4,5,46267,Savings,Dormant,2022-09-18,5602.16,0.00,7.0,10134.44,1447.777143,4.0,1.0,57.142857,14.285714


*Transaction-Analysis*

In [38]:
transaction_chunks = pd.read_sql(
    """
    SELECT
        transaction_id,
        customer_id,
        account_id,
        merchant_id,
        transaction_timestamp,
        transaction_type,
        payment_method,
        channel,
        amount_inr,
        transaction_status,
        fraud_flag,
        fraud_score
    FROM transactions;
    """,
    engine,
    chunksize=100000
)

In [39]:
transaction_list = []

for chunk in transaction_chunks:

    chunk["transaction_timestamp"] = pd.to_datetime(
        chunk["transaction_timestamp"]
    )

    chunk["transaction_date"] = (
        chunk["transaction_timestamp"].dt.date
    )

    chunk["transaction_month"] = (
        chunk["transaction_timestamp"]
        .dt.to_period("M")
        .astype(str)
    )

    chunk["transaction_hour"] = (
        chunk["transaction_timestamp"].dt.hour
    )

    chunk["day_of_week"] = (
        chunk["transaction_timestamp"].dt.day_name()
    )

    chunk["is_weekend"] = (
        chunk["transaction_timestamp"].dt.dayofweek >= 5
    )

    chunk["is_success"] = (
        chunk["transaction_status"] == "Success"
    )

    chunk["is_failed"] = (
        chunk["transaction_status"] == "Failed"
    )

    chunk["is_pending"] = (
        chunk["transaction_status"] == "Pending"
    )

    chunk["is_reversed"] = (
        chunk["transaction_status"] == "Reversed"
    )

    chunk["risk_level"] = pd.cut(
        chunk["fraud_score"],
        bins=[-1, 30, 70, 100],
        labels=["Low", "Medium", "High"]
    )

    chunk["transaction_value_band"] = pd.cut(
        chunk["amount_inr"],
        bins=[-float("inf"), 1000, 5000, 25000, float("inf")],
        labels=[
            "Low Value",
            "Medium Value",
            "High Value",
            "Very High Value"
        ]
    )

    transaction_list.append(chunk)

In [40]:
transaction_analysis = pd.concat(
    transaction_list,
    ignore_index=True
)

In [41]:
transaction_analysis.shape

(1500000, 23)

In [42]:
transaction_analysis.head()

,transaction_id,customer_id,account_id,merchant_id,transaction_timestamp,transaction_type,payment_method,channel,amount_inr,transaction_status,...,transaction_month,transaction_hour,day_of_week,is_weekend,is_success,is_failed,is_pending,is_reversed,risk_level,transaction_value_band
0,2826,55338,36286,29527,2025-10-04 11:09:45.176817,Purchase,UPI,Mobile App,6310.14,Success,...,2025-10,11,Saturday,True,True,False,False,False,Medium,High Value
1,2827,111393,128678,15681,2025-07-07 20:34:12.580092,Transfer,Net Banking,POS,1310.60,Success,...,2025-07,20,Monday,False,True,False,False,False,Medium,Medium Value
2,2828,117928,123674,12455,2025-09-18 01:08:54.187450,Transfer,Net Banking,Web,668.08,Success,...,2025-09,1,Thursday,False,True,False,False,False,Medium,Low Value
3,2829,41130,64210,2208,2025-10-17 18:10:38.958371,Purchase,UPI,Mobile App,193.72,Success,...,2025-10,18,Friday,False,True,False,False,False,Medium,Low Value
4,2830,89295,48485,7439,2025-11-28 09:07:08.959987,Bill Payment,Bank Transfer,Web,1368.12,Success,...,2025-11,9,Friday,False,True,False,False,False,Medium,Medium Value


*Merchant Analysis*

In [43]:
merchants = pd.read_sql(
    """
    SELECT *
    FROM merchants;
    """,
    engine
)

In [44]:
merchant_metrics = pd.read_sql(
    """
    SELECT
        merchant_id,
        COUNT(*) AS total_transactions,
        SUM(amount_inr) AS total_transaction_value,
        AVG(amount_inr) AS average_transaction_value,

        SUM(CASE
            WHEN transaction_status = 'Success'
            THEN 1 ELSE 0
        END) AS successful_transactions,

        SUM(CASE
            WHEN transaction_status = 'Failed'
            THEN 1 ELSE 0
        END) AS failed_transactions,

        SUM(fraud_flag) AS fraud_transactions

    FROM transactions

    GROUP BY merchant_id;
    """,
    engine
)

In [45]:
merchant_analysis = merchants.merge(
    merchant_metrics,
    on="merchant_id",
    how="left"
)

In [46]:
merchant_analysis["success_rate"] = (
    merchant_analysis["successful_transactions"]
    / merchant_analysis["total_transactions"]
    * 100
)

merchant_analysis["failure_rate"] = (
    merchant_analysis["failed_transactions"]
    / merchant_analysis["total_transactions"]
    * 100
)

merchant_analysis["fraud_rate"] = (
    merchant_analysis["fraud_transactions"]
    / merchant_analysis["total_transactions"]
    * 100
)

In [47]:
merchant_analysis.head()

,merchant_id,merchant_name,merchant_category,merchant_city,merchant_state,merchant_tier,onboarding_date,risk_rating,settlement_cycle,total_transactions,total_transaction_value,average_transaction_value,successful_transactions,failed_transactions,fraud_transactions,success_rate,failure_rate,fraud_rate
0,1,Merchant_00001,Grocery,Ahmedabad,Gujarat,Enterprise,2025-03-11,Medium,T+3,42,120668.39,2873.056905,34,2,2,80.952381,4.761905,4.761905
1,2,Merchant_00002,Utilities,Pune,Maharashtra,SMB,2022-03-13,Low,T+2,49,169912.66,3467.605306,39,2,1,79.591837,4.081633,2.040816
2,3,Merchant_00003,Utilities,Jaipur,Rajasthan,SMB,2025-02-19,Low,T+3,43,138043.52,3210.314419,37,2,0,86.046512,4.651163,0.000000
3,4,Merchant_00004,Healthcare,Indore,Madhya Pradesh,SMB,2023-07-05,Low,T+2,46,136655.35,2970.768478,37,4,0,80.434783,8.695652,0.000000
4,5,Merchant_00005,Education,Jaipur,Rajasthan,SMB,2025-03-05,Low,T+2,37,137062.30,3704.386486,26,7,1,70.270270,18.918919,2.702703


*ChargeBack Analysis*

In [48]:
chargebacks = pd.read_sql(
    """
    SELECT *
    FROM chargebacks;
    """,
    engine
)

In [49]:
chargeback_analysis = chargebacks.copy()

In [50]:
chargeback_analysis["unrecovered_amount"] = (
    chargeback_analysis["chargeback_amount_inr"]
    - chargeback_analysis["recovery_amount_inr"]
)

In [51]:
chargeback_analysis["recovery_rate"] = (
    chargeback_analysis["recovery_amount_inr"]
    / chargeback_analysis["chargeback_amount_inr"]
    * 100
)

In [52]:
chargeback_analysis["recovery_status"] = np.select(
    [
        chargeback_analysis["recovery_amount_inr"] == 0,

        chargeback_analysis["recovery_amount_inr"]
        < chargeback_analysis["chargeback_amount_inr"],

        chargeback_analysis["recovery_amount_inr"]
        >= chargeback_analysis["chargeback_amount_inr"]
    ],
    [
        "No Recovery",
        "Partial Recovery",
        "Full Recovery"
    ],
    default="Unknown"
)

In [53]:
chargeback_analysis.head()

,chargeback_id,transaction_id,chargeback_date,reason_code,chargeback_amount_inr,chargeback_status,recovery_amount_inr,merchant_response_days,unrecovered_amount,recovery_rate,recovery_status
0,1,278913,2026-04-29,Processing Error,2200.99,Won,1520.16,8,680.83,69.067102,Partial Recovery
1,2,1121236,2026-01-19,Fraud,7864.47,Lost,2752.50,24,5111.97,34.999180,Partial Recovery
2,3,958720,2025-06-14,Product Not Received,689.04,Won,220.31,21,468.73,31.973470,Partial Recovery
3,4,1108194,2026-01-02,Fraud,2764.61,Lost,96.71,1,2667.90,3.498143,Partial Recovery
4,5,993114,2024-09-23,Processing Error,202.21,Lost,28.60,7,173.61,14.143712,Partial Recovery


*Support Analysis *

In [54]:
support = pd.read_sql(
    """
    SELECT *
    FROM support_tickets;
    """,
    engine
)

In [55]:
support_analysis = support.copy()

In [56]:
support_analysis["created_at"] = pd.to_datetime(
    support_analysis["created_at"]
)

In [57]:
support_analysis["ticket_month"] = (
    support_analysis["created_at"]
    .dt.to_period("M")
    .astype(str)
)

support_analysis["ticket_day"] = (
    support_analysis["created_at"]
    .dt.day_name()
)

In [58]:
support_analysis.head()

,ticket_id,customer_id,created_at,issue_category,priority,channel,ticket_status,resolution_hours,customer_satisfaction,agent_team,ticket_month,ticket_day
0,1,8522,2023-09-17 19:32:58,Fraud Alert,High,Phone,Resolved,4.56,5.0,Payments,2023-09,Sunday
1,2,98944,2025-02-02 10:35:10,Payment Failure,High,Web,Closed,6.39,4.0,Fraud & Risk,2025-02,Sunday
2,3,31645,2025-12-26 07:04:20,Refund,High,Mobile App,Closed,14.39,3.0,Fraud & Risk,2025-12,Friday
3,4,75575,2025-08-04 22:15:06,KYC,High,Email,Closed,12.37,4.0,Disputes,2025-08,Monday
4,5,25145,2024-06-19 14:05:37,Fraud Alert,Low,Phone,Resolved,10.68,4.0,Fraud & Risk,2024-06,Wednesday


In [59]:
import os

os.makedirs("analytical_data", exist_ok=True)

In [60]:
customer_analysis.to_csv(
    "analytical_data/customer_analysis.csv",
    index=False
)

account_analysis.to_csv(
    "analytical_data/account_analysis.csv",
    index=False
)

transaction_analysis.to_csv(
    "analytical_data/transaction_analysis.csv",
    index=False
)

merchant_analysis.to_csv(
    "analytical_data/merchant_analysis.csv",
    index=False
)

chargeback_analysis.to_csv(
    "analytical_data/chargeback_analysis.csv",
    index=False
)

support_analysis.to_csv(
    "analytical_data/support_analysis.csv",
    index=False
)

print("All 6 analytical files saved successfully.")

All 6 analytical files saved successfully.


ANALYTICAL TABLE SHAPE VALIDATION

In [61]:
tables_summary = {
    "Customer": customer_analysis.shape,
    "Account": account_analysis.shape,
    "Transaction": transaction_analysis.shape,
    "Merchant": merchant_analysis.shape,
    "Chargeback": chargeback_analysis.shape,
    "Support": support_analysis.shape
}

tables_summary

{'Customer': (120000, 20),
 'Account': (180000, 14),
 'Transaction': (1500000, 23),
 'Merchant': (35000, 18),
 'Chargeback': (24000, 11),
 'Support': (90000, 12)}

Duplicate Primary Key Check

In [62]:
print("Customers:", customer_analysis["customer_id"].duplicated().sum())
print("Accounts:", account_analysis["account_id"].duplicated().sum())
print("Transactions:", transaction_analysis["transaction_id"].duplicated().sum())
print("Merchants:", merchant_analysis["merchant_id"].duplicated().sum())

Customers: 0
Accounts: 0
Transactions: 0
Merchants: 0


ChargeBack Calculation Checking

In [63]:
invalid_recovery = (
    chargeback_analysis["recovery_amount_inr"] >
    chargeback_analysis["chargeback_amount_inr"]
).sum()

print("Invalid recovery records:", invalid_recovery)

Invalid recovery records: 0


In [64]:
customer_analysis.isnull().sum()

customer_id                      0
customer_tenure_months           0
age                              0
city                             0
state                            0
customer_segment                 0
income_band                      0
kyc_status                       0
acquisition_channel              0
signup_date                      0
risk_band                        0
total_transactions           26787
total_transaction_value      26787
average_transaction_value    26787
successful_transactions      26787
failed_transactions          26787
fraud_transactions           26787
transaction_success_rate     26787
transaction_failure_rate     26787
fraud_rate                   26787
dtype: int64

Zero Transactions Customers Validation

In [65]:
zero_transaction_customers = customer_analysis[
    customer_analysis["total_transactions"].isna()
]

print(
    "Customers with no transaction activity:",
    len(zero_transaction_customers)
)

Customers with no transaction activity: 26787


Overall Null Summary For Analytical Tables

In [66]:
print("CUSTOMER ANALYSIS")
print(customer_analysis.isnull().sum().sort_values(ascending=False))

print("\nACCOUNT ANALYSIS")
print(account_analysis.isnull().sum().sort_values(ascending=False))

print("\nTRANSACTION ANALYSIS")
print(transaction_analysis.isnull().sum().sort_values(ascending=False))

print("\nMERCHANT ANALYSIS")
print(merchant_analysis.isnull().sum().sort_values(ascending=False))

print("\nCHARGEBACK ANALYSIS")
print(chargeback_analysis.isnull().sum().sort_values(ascending=False))

print("\nSUPPORT TICKET ANALYSIS")
print(support_analysis.isnull().sum().sort_values(ascending=False))

CUSTOMER ANALYSIS
fraud_transactions           26787
transaction_success_rate     26787
transaction_failure_rate     26787
total_transactions           26787
fraud_rate                   26787
successful_transactions      26787
average_transaction_value    26787
total_transaction_value      26787
failed_transactions          26787
age                              0
customer_tenure_months           0
customer_id                      0
city                             0
risk_band                        0
acquisition_channel              0
signup_date                      0
customer_segment                 0
income_band                      0
state                            0
kyc_status                       0
dtype: int64

ACCOUNT ANALYSIS
failed_transactions          50
success_rate                 50
successful_transactions      50
average_transaction_value    50
total_transaction_value      50
total_transactions           50
failure_rate                 50
opened_date                

Primary Key NULL'S

In [68]:
account_analysis["account_id"].isnull().sum()

transaction_analysis["transaction_id"].isnull().sum()

merchant_analysis["merchant_id"].isnull().sum()

chargeback_analysis["chargeback_id"].isnull().sum()

support_analysis["ticket_id"].isnull().sum()

np.int64(0)

Transaction Analysis Null Values

In [69]:
transaction_analysis[
    [
        "transaction_id",
        "customer_id",
        "account_id",
        "merchant_id",
        "amount_inr",
        "payment_method",
        "transaction_status",
        "fraud_score"
    ]
].isnull().sum()

transaction_id        0
customer_id           0
account_id            0
merchant_id           0
amount_inr            0
payment_method        0
transaction_status    0
fraud_score           0
dtype: int64

Merchant Analysis Null Values

In [70]:
merchant_analysis[
    [
        "merchant_id",
        "total_transactions",
        "total_transaction_value",
        "average_transaction_value",
        "success_rate",
        "failure_rate",
        "fraud_rate"
    ]
].isnull().sum()

merchant_id                  0
total_transactions           0
total_transaction_value      0
average_transaction_value    0
success_rate                 0
failure_rate                 0
fraud_rate                   0
dtype: int64

ChargeBack Analysis Null Values


In [71]:
chargeback_analysis[
    [
        "chargeback_id",
        "transaction_id",
        "chargeback_amount_inr",
        "recovery_amount_inr",
        "unrecovered_amount",
        "recovery_rate"
    ]
].isnull().sum()

chargeback_id            0
transaction_id           0
chargeback_amount_inr    0
recovery_amount_inr      0
unrecovered_amount       0
recovery_rate            0
dtype: int64